# Семинар 2 (Offline): Сверточные нейронные сети (CNN) в PyTorch

### Введение: Почему обычные (полносвязные) сети плохо подходят для картинок?
На прошлом семинаре мы работали с табличными точками, где у каждого объекта было всего 2 признака. Но что, если мы хотим научить нейросеть распознавать изображения?

Представим небольшую цветную фотографию размером **100 × 100 пикселей** (по 3 канала RGB на пиксель). Если вытянуть её в один длинный вектор, получится **30 000 входных чисел**.
Если подать этот вектор в обычный полносвязный слой `nn.Linear(30000, 1000)`, то только в первом слое будет **30 миллионов весов**! Это порождает две огромные проблемы:
1. **Слишком много параметров:** Сеть моментально переобучится (просто зазубрит обучающие картинки) и потребует колоссальной видеопамяти.
2. **Потеря пространственной структуры:** При вытягивании картинки в строчку соседние по вертикали пиксели оказываются далеко друг от друга. Кроме того, если сдвинуть объект на картинке на 2 пикселя вправо, для полносвязной сети это будет совершенно новый набор чисел.

**Решение — Сверточные нейронные сети (Convolutional Neural Networks, CNN).** Вместо того чтобы смотреть на всю картинку целиком, сверточный слой скользит по изображению маленьким окошком (фильтром размером, например, `3 × 3`) и ищет локальные узоры: сначала границы и уголки, на следующих слоях — кружочки и дуги, а глубоко внутри сети — целые цифры или объекты.

---
### Часть 0. Подключение GPU и проверка вычислительного устройства
Сверточные сети выполняют тысячи операций скользящего окна для каждого изображения в батче, поэтому при работе с большими картинками видеокарта (`CUDA` или `MPS`) ускоряет обучение в 10–50 раз.

**Правило переноса для CNN:**
* Саму сверточную модель мы переносим один раз при создании: `model = DigitsCNN().to(device)`.
* Каждый батч картинок `images` и меток `labels` мы переносим на `device` внутри цикла обучения: `images, labels = images.to(device), labels.to(device)`.
* Перед визуализацией картинок или фильтров через `matplotlib` мы возвращаем тензор в оперативную память процессора: `.cpu()`.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Фиксируем зерно генератора случайных чисел
torch.manual_seed(42)
np.random.seed(42)

# Автоматический выбор лучшего доступного устройства (NVIDIA CUDA -> Apple MPS -> CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Используется видеокарта NVIDIA GPU: {torch.cuda.get_device_name(0)}")
    print(f"Доступно видеопамяти: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Используется ускоритель Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("Видеокарта не обнаружена. Используется процессор (CPU) — для нашего компактного датасета его хватит с запасом!")

---
### Часть 1. Как устроены свертка (`Conv2d`) и пулинг (`MaxPool2d`)?
Любая классическая сверточная сеть чередует два главных строительных блока:

1. **Сверточный слой `nn.Conv2d(in_channels, out_channels, kernel_size, padding)`:**
   * `in_channels` — глубина входного изображения (для черно-белой картинки `1`, для цветной RGB — `3`).
   * `out_channels` — сколько разных фильтров (детекторов признаков) мы хотим обучить. Каждый фильтр создает свою «карту признаков» (*feature map*).
   * `kernel_size` — размер скользящего окошка (обычно `3`, то есть матрица `3 × 3`).
   * `padding` — рамка из нулей вокруг картинки. Если задать `padding=1` для фильтра `3 × 3`, размер картинки после свертки не уменьшится.

2. **Слой подвыборки (пулинга) `nn.MaxPool2d(kernel_size=2)`:**
   * Разбивает карту признаков на квадратики `2 × 2` и из каждого квадратика оставляет только **максимальное** число.
   * **Зачем это нужно?** Картинка сжимается в 2 раза по ширине и высоте (в 4 раза по площади), вычисления ускоряются, а сеть становится устойчивой к небольшим сдвигам объекта.

#### Формат 4D-тензоров для изображений в PyTorch: `(B, C, H, W)`
В PyTorch картинки всегда передаются в виде четырехмерного тензора:
* **B (Batch size):** количество картинок в одном батче (например, `32`).
* **C (Channels):** количество цветовых каналов (`1` для ч/б, `3` для RGB).
* **H (Height):** высота картинки в пикселях.
* **W (Width):** ширина картинки в пикселях.

In [ ]:
# Имитируем батч из 4 черно-белых картинок размером 8x8 пикселей -> форма (4, 1, 8, 8)
fake_images = torch.randn(4, 1, 8, 8, device=device)

# Создаем сверточный слой: 1 входной канал -> 8 фильтров размера 3x3 с padding=1
conv_layer = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1).to(device)
pool_layer = nn.MaxPool2d(kernel_size=2)

# Пропускаем батч через свертку и пулинг
after_conv = conv_layer(fake_images)
after_pool = pool_layer(after_conv)

print("1. Исходная форма батча (B, C, H, W):", fake_images.shape)
print("2. После Conv2d (padding=1 сохранил размер 8x8, но стало 8 каналов):", after_conv.shape)
print("3. После MaxPool2d(2) (ширина и высота уменьшились вдвое до 4x4):", after_pool.shape)

#### Мини-эксперимент №1: Зачем нужен `padding`?
* **Задание:** В ячейке выше измените `padding=1` на `padding=0` в слое `nn.Conv2d` и перезапустите ячейку.
* **Что вы увидите и почему:** Форма `after_conv` станет `(4, 8, 6, 6)` вместо `(4, 8, 8, 8)`. Когда окошко `3 × 3` скользит по сетке `8 × 8` без рамки из нулей, оно не может встать центром на крайние пиксели, поэтому картинка «съедается» по 1 пикселю с каждой стороны (`8 - 2 = 6`). Для маленьких картинок `8 × 8` это критично, поэтому `padding=1` помогает сохранить размер.

---
### Часть 2. Полностью автономный датасет рукописных цифр (`load_digits`)
Чтобы наш семинар работал **полностью без подключения к интернету** и не требовал скачивания тяжелых архивов, мы воспользуемся встроенным в библиотеку `scikit-learn` датасетом **`load_digits`**.

**Что внутри датасета?**
* **1797** реальных черно-белых изображений рукописных цифр от `0` до `9` (10 классов).
* Размер каждой картинки — **8 × 8 пикселей**, яркость каждого пикселя задана числом от `0` до `16`.
* Мы разделим данные на **1437 обучающих** картинок (`train`) и **360 тестовых** (`test`), отмасштабируем яркость в диапазон `[0, 1]` (разделив на `16.0`) и упакуем в стандартные `TensorDataset` и `DataLoader`.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# 1. Загружаем датасет локально из памяти sklearn (интернет не требуется!)
digits = load_digits()
X_raw = digits.images   # Массив формы (1797, 8, 8)
y_raw = digits.target   # Метки классов от 0 до 9, форма (1797,)

print("Всего изображений:", X_raw.shape[0], "| Размер каждой картинки:", X_raw.shape[1:])
print("Классы цифр:", np.unique(y_raw))

# 2. Нормализуем яркость пикселей из [0, 16] в [0.0, 1.0] — нейросети лучше учатся на числах около 0..1
X_norm = X_raw / 16.0

# 3. Делим на обучающую (80%) и тестовую (20%) выборки со стратификацией (равная доля каждой цифры)
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# 4. Добавляем ось каналов C=1 с помощью .unsqueeze(1), чтобы получить 4D-тензор (N, 1, 8, 8)
X_train_t = torch.FloatTensor(X_train).unsqueeze(1)
y_train_t = torch.LongTensor(y_train)  # Для функции потерь CrossEntropyLoss метки должны быть целыми (LongTensor)

X_test_t = torch.FloatTensor(X_test).unsqueeze(1)
y_test_t = torch.LongTensor(y_test)

# 5. Создаем DataLoader с батчами по 32 картинки
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64, shuffle=False)

print(f"Обучающая выборка: {X_train_t.shape} | Тестовая выборка: {X_test_t.shape}")

# 6. Визуализируем первые 10 картинок из обучающей выборки
fig, axes = plt.subplots(1, 10, figsize=(13, 2))
for i in range(10):
    axes[i].imshow(X_train[i], cmap='gray_r')  # gray_r: темные чернила на светлом фоне
    axes[i].set_title(f"Цифра: {y_train[i]}", fontsize=10)
    axes[i].axis('off')
plt.suptitle("Примеры изображений из датасета load_digits (8x8 пикселей)", y=1.08)
plt.show()

---
### Часть 3. Сборка архитектуры сверточной нейросети (`DigitsCNN`)
Любая классическая CNN логически делится на две части:
1. **Экстрактор признаков (`self.features`):** последовательность сверток (`Conv2d`), функций активации (`ReLU`) и пулингов (`MaxPool2d`). Проследим за изменением формы тензора для одной картинки `(1, 8, 8)`:
   * Вход: `(1, 8, 8)`
   * `Conv2d(1 -> 16, kernel_size=3, padding=1)` + `ReLU`: получаем `(16, 8, 8)` (16 карт признаков).
   * `MaxPool2d(2)`: сжимаем вдвое до `(16, 4, 4)`.
   * `Conv2d(16 -> 32, kernel_size=3, padding=1)` + `ReLU`: получаем `(32, 4, 4)` (32 более сложных признака).
   * `MaxPool2d(2)`: сжимаем вдвое до `(32, 2, 2)`.
2. **Классификатор (`self.classifier`):**
   * Слой `nn.Flatten()` вытягивает трехмерный кубик признаков `(32, 2, 2)` в одномерный вектор длиной `32 * 2 * 2 = 128` чисел.
   * Полносвязный слой `nn.Linear(128, 64)` + `ReLU` + `nn.Dropout(0.25)` (слой `Dropout` случайно отключает 25% нейронов во время обучения, чтобы сеть не переобучалась).
   * Выходной слой `nn.Linear(64, 10)` выдает 10 чисел (логитов) — по одному на каждую цифру от `0` до `9`.

> **Важно про `nn.CrossEntropyLoss()`:** В отличие от бинарной классификации, где мы ставили `Sigmoid()` в конце сети, для многоклассовой классификации в PyTorch функцию `Softmax` в конец модели **не ставят**. Функция потерь `nn.CrossEntropyLoss()` сама применяет `LogSoftmax` внутри себя — это работает быстрее и математически устойчивее.

In [ ]:
class DigitsCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Блок 1: Извлечение пространственных признаков (свертки + пулинг)
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # (16, 8, 8) -> (16, 4, 4)
            
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)   # (32, 4, 4) -> (32, 2, 2)
        )
        
        # Блок 2: Классификация на основе найденных признаков
        self.classifier = nn.Sequential(
            nn.Flatten(),                 # (B, 32, 2, 2) -> (B, 128)
            nn.Linear(32 * 2 * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.25),             # Защита от переобучения
            nn.Linear(64, num_classes)    # 10 выходов (по одному на цифру 0..9)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Создаем модель и переносим её на GPU (если доступен) или CPU
model = DigitsCNN(num_classes=10).to(device)
print(model)

# Подсчитаем общее количество обучаемых весов в нашей CNN
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nВсего обучаемых параметров в сети: {total_params:,}")

---
### Часть 4. Обучение модели (`model.train()`) и оценка на тесте (`model.eval()`)
При обучении сетей, в которых есть слои `Dropout` или `BatchNorm`, обязательно нужно переключать режимы работы модели:
* **`model.train()`** — включаем перед циклом обучения. Слой `Dropout` активно работает и случайно отключает часть нейронов для регуляризации.
* **`model.eval()`** — включаем перед проверкой точности на тестовых данных. Слой `Dropout` отключается, и сеть использует все 100% своих нейронов для максимально точного предсказания.

Так как датасет компактный, обучение на 25 эпох займет всего **2–4 секунды** даже на обычном CPU (и доли секунды на GPU).

In [ ]:
torch.manual_seed(42)
model = DigitsCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

epochs = 25
train_loss_history = []
test_acc_history = []

for epoch in range(epochs):
    # --- ЭТАП 1: ОБУЧЕНИЕ ---
    model.train()  # Включаем режим обучения (активирует Dropout)
    running_loss = 0.0
    
    for images, labels in train_loader:
        # Переносим батч картинок и правильных ответов на устройство (GPU/CPU)
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)              # Прямой проход
        loss = criterion(outputs, labels)    # Считаем ошибку
        
        optimizer.zero_grad()                # Обнуляем градиенты
        loss.backward()                      # Обратный проход
        optimizer.step()                     # Обновляем веса
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    train_loss_history.append(epoch_loss)
    
    # --- ЭТАП 2: ПРОВЕРКА НА ТЕСТЕ ---
    model.eval()   # Включаем режим оценки (отключает Dropout)
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # Выбираем класс с максимальным выходным значением (логитом)
            _, predicted = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    epoch_acc = correct / total
    test_acc_history.append(epoch_acc)

print(f"Обучение завершено! Итоговая точность на тестовой выборке: {test_acc_history[-1]:.2%}")

# Строим графики обучения
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_loss_history, color='crimson', linewidth=2)
ax1.set_title("Падение ошибки на обучении (Train Loss)")
ax1.set_xlabel("Эпоха")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot([acc * 100 for acc in test_acc_history], color='forestgreen', linewidth=2)
ax2.set_title("Рост точности на тесте (Test Accuracy)")
ax2.set_xlabel("Эпоха")
ax2.set_ylabel("Точность (%)")
ax2.grid(True, linestyle='--', alpha=0.5)

plt.show()

---
### Часть 5. Визуализация предсказаний, разбор ошибок и «взгляд внутрь» фильтров
Посмотрим, как наша обученная сеть справляется с тестовыми цифрами, на каких сложных почерках она всё же ошибается, и как выглядят сами сверточные фильтры `3 × 3`, которые сеть выучила с нуля.

In [ ]:
model.eval()
with torch.no_grad():
    all_outputs = model(X_test_t.to(device))
    all_preds = torch.argmax(all_outputs, dim=1).cpu().numpy()

# 1. Отрисуем 12 случайных примеров из тестовой выборки
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for idx, ax in enumerate(axes.flat):
    ax.imshow(X_test[idx], cmap='gray_r')
    pred_label = all_preds[idx]
    true_label = y_test[idx]
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f"Предск: {pred_label} (Ист: {true_label})", color=color, fontsize=10)
    ax.axis('off')
plt.suptitle("Примеры работы CNN на тестовой выборке", y=1.02)
plt.tight_layout()
plt.show()

# 2. Найдем и покажем те редкие картинки, на которых модель ошиблась!
error_indices = np.where(all_preds != y_test)[0]
print(f"Всего ошибок на 360 тестовых картинках: {len(error_indices)}")

if len(error_indices) > 0:
    num_show = min(len(error_indices), 8)
    fig, axes = plt.subplots(1, num_show, figsize=(12, 2.2))
    if num_show == 1:
        axes = [axes]
    for i in range(num_show):
        err_idx = error_indices[i]
        axes[i].imshow(X_test[err_idx], cmap='gray_r')
        axes[i].set_title(f"Сеть: {all_preds[err_idx]}\nФакт: {y_test[err_idx]}", color='crimson', fontsize=9)
        axes[i].axis('off')
    plt.suptitle("Разбор ошибок модели (сложный почерк)", y=1.12)
    plt.show()

In [ ]:
# 3. Заглянем внутрь первого сверточного слоя: визуализируем 16 обученных фильтров 3x3
# Достаем матрицу весов первого слоя (форма: [16, 1, 3, 3]) и переносим на CPU
first_conv_weights = model.features[0].weight.detach().cpu().numpy()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for i, ax in enumerate(axes.flat):
    # Отрисовываем i-й фильтр размером 3x3
    ax.imshow(first_conv_weights[i, 0], cmap='coolwarm')
    ax.set_title(f"Фильтр {i+1}", fontsize=9)
    ax.axis('off')
plt.suptitle("16 выученных сверточных фильтров (3x3) первого слоя", y=1.04)
plt.tight_layout()
plt.show()

#### Мини-эксперимент №2: «Поменяй и посмотри» (для закрепления темы CNN)
Предложите участникам семинара поэкспериментировать с архитектурой `DigitsCNN` (в **Части 3** и **Части 4**):

1. **Эксперимент А: Влияние размера ядра свертки (`kernel_size`)**
   * *Что сделать:* В первом слое `nn.Conv2d(1, 16, kernel_size=3, padding=1)` замените `kernel_size=3, padding=1` на `kernel_size=5, padding=2` (обратите внимание: чтобы сохранить размер картинки при ядре `5 × 5`, рамка `padding` должна быть равна `2`). Перезапустите обучение.
   * *Наблюдение:* Как изменилось общее количество обучаемых параметров в сети? (Каждый из 16 фильтров теперь хранит `25` чисел вместо `9`).

2. **Эксперимент Б: Что будет без пулинга (`MaxPool2d`)?**
   * *Что сделать:* Закомментируйте оба слоя `nn.MaxPool2d(kernel_size=2)` в `self.features`. Обратите внимание: так как картинка перестанет сжиматься с `8 × 8` до `2 × 2`, вход в первый полносвязный слой в `self.classifier` нужно изменить с `nn.Linear(32 * 2 * 2, 64)` на `nn.Linear(32 * 8 * 8, 64)`.
   * *Наблюдение:* Посмотрите на строку `Всего обучаемых параметров в сети` — число параметров вырастет в несколько раз, а обучение станет медленнее при почти той же итоговой точности (~98%). Это наглядно доказывает, зачем в CNN нужен пулинг.